# 16 · HMDA：神经网络（TabM-mini）与 CatBoost 融合（Colab GPU）

**目的**：现有最终提交（15 的 `B_seeds`）全部是 CatBoost，彼此预测相关 > 0.98，互相平均能抵消的错误有限；LightGBM 同样是树模型（相关 0.979）且单独太弱，所以没有帮助。神经网络的学习方式完全不同，**如果它单独不太弱（约 0.915 以上）**，与 CatBoost 融合才有机会互补。

**模型**：TabM-mini（Gorishniy 等，2024 的简化版，PyTorch 自己实现，不依赖外部包）。32 个「成员」共享同一个 3 层 MLP，每个成员只在输入端有自己的 ±1 缩放向量；预测取 32 个成员概率的平均，相当于一次训练得到一个小集成。

**输入**（与最终 CatBoost 相同的 G7，72 列），全部只在外层训练部分上拟合：
- 数值列：分位数正态化 + 16 段分段线性编码（PLE）+ 缺失标记；
- 类别列：词表 + 8 维嵌入，出现少于 5 次或未见过的类别归入「未知」。

**协议**：固定 seed=42 五折；外层训练部分再留出 15% 做早停（按 AUC，耐心 8 个 epoch），**不再在整个外层训练部分重训**（神经网络的轮数外推不可靠）。

**流程**
1. 默认 `FOLDS = [1]`：只跑第 1 折（预计 5–15 分钟），最后一节直接给出与 CatBoost（15 的 B_seeds）在同一折上的融合效果和建议。
2. 若第 1 折融合提升 ≥ 0.0003，把 `FOLDS` 改成 `[1, 2, 3, 4, 5]` 从头运行（第 1 折自动跳过），最后一节会写出三个权重的提交文件。
3. 否则到此为止，最终提交不变。

需要 15 已经完成（读取 `artifacts/final_gpu/round2_final_v1/` 中 9 个 CatBoost 组的五折预测）。Colab 选 **T4 GPU**。

## 1. 环境、GPU 与配置

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'HMDA_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis',
)).expanduser().resolve()
PREPARED_DIR = PROJECT_DIR / 'artifacts' / 'prepared'
SEED = 42
CATBOOST_VERSION = '1.2.10'
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启 Colab 运行时后从头运行')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'catboost=={CATBOOST_VERSION}',
    ])
    importlib.invalidate_caches()

import catboost
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('未找到 nvidia-smi：请在 Colab 更改运行时类型，选择 T4 GPU 后从头运行')
gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True, capture_output=True, check=False,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError('NVIDIA GPU 不可用：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = gpu_probe.stdout.strip().splitlines()[0]

import torch
import torch.nn as nn
from sklearn.preprocessing import QuantileTransformer

EXPERIMENT_ID = 'tabm_mini_v1'
FOLDS = [1]                 # 先只跑第 1 折试水；值得继续再改为 [1, 2, 3, 4, 5]
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'nn_models' / EXPERIMENT_ID
CATBOOST_DIR = PROJECT_DIR / 'artifacts' / 'final_gpu' / 'round2_final_v1'   # 13/15 的结果
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

NO_FREQ = ['dti', 'ltv', 'income', 'missing']
G7 = NO_FREQ + ['precision', 'thresholds', 'implied', 'term']   # 与最终 CatBoost 相同的 72 列
NN_CONFIG = dict(
    k=32,                   # TabM 的「集成成员」数：32 个共享大部分权重的 MLP
    width=512, depth=3, dropout=0.1,
    cat_dim=8,              # 每个类别列的嵌入维度
    ple_bins=16,            # 数值列分段线性编码的分段数
    rare_min_count=5,       # 外层训练部分出现少于 5 次的类别并入「未知」
    lr=1e-3, weight_decay=1e-4, batch_size=512,
    max_epochs=60, patience=8,
    inner_stop_fraction=0.15,
)
BLEND_WEIGHTS = [0.1, 0.2, 0.3]
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'项目：{PROJECT_DIR}')
print(f'设备：{DEVICE}（{GPU_NAME}）；torch {torch.__version__}；折：{FOLDS}；输出：{OUTPUT_DIR}')

Mounted at /content/drive
项目：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis
设备：cuda（Tesla T4, 15360 MiB）；torch 2.11.0+cu130；折：[1]；输出：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis/artifacts/nn_models/tabm_mini_v1


## 2. 回读并核对 01 的数据契约（与 04 相同）

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_hmda_data_preparation.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = PROJECT_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(PROJECT_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    PROJECT_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    PROJECT_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

数据核对通过：训练 105,000，测试 45,000，56 个原始特征。


accepted,0,1
fold,,
1,3151,17849
2,3151,17849
3,3151,17849
4,3151,17849
5,3152,17848


## 3. DTI 上下界（与 04 相同）

In [3]:
DTI_BOUNDS = {
    **{str(value): (float(value), float(value)) for value in range(36, 50)},
    '20%-<30%': (20.0, 30.0),
    '30%-<36%': (30.0, 36.0),
    '50%-60%': (50.0, 60.0),
    '<20%': (np.nan, 20.0),
    '>60%': (60.0, np.nan),
    'MISSING': (np.nan, np.nan),
}


def add_dti_bounds(frame):
    unknown = set(frame['debt_to_income_ratio'].unique()) - set(DTI_BOUNDS)
    require(not unknown, f'DTI 出现未知类别：{sorted(map(str, unknown))}')
    result = frame.copy()
    pairs = frame['debt_to_income_ratio'].map(DTI_BOUNDS)
    result['dti_lower'] = pairs.map(lambda pair: pair[0]).astype('float64')
    result['dti_upper'] = pairs.map(lambda pair: pair[1]).astype('float64')
    require(result.columns.tolist() == FEATURE_COLUMNS + ['dti_lower', 'dti_upper'],
            'DTI 特征列或顺序错误')
    require(result['debt_to_income_ratio'].equals(frame['debt_to_income_ratio']),
            '原始 DTI 类别被修改')
    return result


X_dti = add_dti_bounds(X_base)
X_test_dti = add_dti_bounds(X_test_base)
print(f'候选特征：原始 56 列 + 2 列 DTI 上下界 = {X_dti.shape[1]} 列。')
display(X_dti[['debt_to_income_ratio', 'dti_lower', 'dti_upper']].head())

候选特征：原始 56 列 + 2 列 DTI 上下界 = 58 列。


,debt_to_income_ratio,dti_lower,dti_upper
0,20%-<30%,20.0,30.0
1,49,49.0,49.0
2,20%-<30%,20.0,30.0
3,44,44.0,44.0
4,<20%,NaN,20.0


## 4. 第一轮特征块（与 10 相同）

In [4]:
def safe_ratio(numerator, denominator):
    valid = numerator.notna() & denominator.notna() & (denominator > 0)
    return (numerator / denominator.where(valid)).where(valid).astype('float64')


def build_blocks(train, test):
    both = pd.concat([train, test], ignore_index=True)
    blocks = {}
    blocks['dti'] = add_dti_bounds(both)[['dti_lower', 'dti_upper']]

    ltv_calc = 100 * safe_ratio(both['loan_amount'], both['property_value'])
    blocks['ltv'] = pd.DataFrame({
        'ltv_calc': ltv_calc,
        'ltv_gap': both['loan_to_value_ratio'] - ltv_calc,
    })

    income_dollars = both['income'] * 1000      # HMDA 收入单位为千美元
    blocks['income'] = pd.DataFrame({
        'income_to_area': safe_ratio(income_dollars, both['ffiec_msa_md_median_family_income']),
        'loan_to_income': safe_ratio(both['loan_amount'], income_dollars),
        'value_to_income': safe_ratio(both['property_value'], income_dollars),
    })

    missing = pd.DataFrame({
        'ltv': both['loan_to_value_ratio'].isna(),
        'pv': both['property_value'].isna(),
        'inc': both['income'].isna(),
        'term': both['loan_term'].isna(),
        'dti': both['debt_to_income_ratio'].eq('MISSING'),
        'age': both['applicant_age'].eq('MISSING'),
    })
    blocks['missing'] = pd.DataFrame({
        'missing_pattern': 'm' + missing.astype(int).astype(str).agg(''.join, axis=1),
        'missing_count': missing.sum(axis=1).astype('float64'),
    })

    freq = {}
    for column in ['county_code', 'derived_msa-md', 'state_code']:
        counts = both[column].map(both[column].value_counts()).astype('float64')
        freq[f'freq_{column}'] = counts.where(both[column].ne('MISSING'))
    blocks['freq'] = pd.DataFrame(freq)

    n_train = len(train)
    out = {}
    for name, frame in blocks.items():
        require(len(frame) == len(both), f'{name} 行数不符')
        out[name] = (frame.iloc[:n_train].reset_index(drop=True),
                     frame.iloc[n_train:].reset_index(drop=True))
    return out


BLOCKS = build_blocks(X_base, X_test_base)
NEW_CATEGORICAL = {'missing_pattern'}


def variant_frames(blocks):
    train_parts = [X_base] + [BLOCKS[b][0] for b in blocks]
    test_parts = [X_test_base] + [BLOCKS[b][1] for b in blocks]
    train = pd.concat(train_parts, axis=1)
    test = pd.concat(test_parts, axis=1)
    require(train.columns.is_unique and train.columns.equals(test.columns), '变体列不一致')
    require(train.iloc[:, :len(FEATURE_COLUMNS)].equals(X_base), '原始 56 列被修改')
    cats = CATEGORICAL_COLUMNS + [c for c in train.columns if c in NEW_CATEGORICAL]
    return train, test, cats


for name, (train_block, _) in BLOCKS.items():
    described = train_block.describe(include='all').T
    print(f'\n[{name}] 缺失率：', train_block.isna().mean().round(3).to_dict())
    display(described.iloc[:, :6])


[dti] 缺失率： {'dti_lower': 0.117, 'dti_upper': 0.117}


,count,mean,std,min,25%,50%
dti_lower,92671.0,39.680947,10.654997,20.0,30.0,42.0
dti_upper,92730.0,41.033290,9.699260,20.0,36.0,40.0



[ltv] 缺失率： {'ltv_calc': 0.045, 'ltv_gap': 0.118}


,count,mean,std,min,25%,50%
ltv_calc,100244.0,82.070292,54.826068,0.302115,74.683544,88.235294
ltv_gap,92643.0,5.568211,439.329812,-7717.790000,-0.719136,0.000000



[income] 缺失率： {'income_to_area': 0.054, 'loan_to_income': 0.053, 'value_to_income': 0.088}


,count,mean,std,min,25%,50%
income_to_area,99343.0,1.563771,3.274112,-6.550349,0.746269,1.108830
loan_to_income,99468.0,2.891506,4.565102,0.002606,1.810560,2.692308
value_to_income,95792.0,3.713296,6.426108,0.009722,2.378641,3.288576



[missing] 缺失率： {'missing_pattern': 0.0, 'missing_count': 0.0}


,count,unique,top,freq,mean,std
missing_pattern,105000,42,m000000,88200,NaN,NaN
missing_count,105000.0,NaN,NaN,NaN,0.336019,0.956439



[freq] 缺失率： {'freq_county_code': 0.013, 'freq_derived_msa-md': 0.0, 'freq_state_code': 0.008}


,count,mean,std,min,25%,50%
freq_county_code,103661.0,364.801787,470.338073,1.0,67.0,200.0
freq_derived_msa-md,105000.0,4000.393038,7326.553843,2.0,353.0,927.0
freq_state_code,104176.0,6209.485976,4859.097575,2.0,2667.0,4441.0


## 5. 第二轮特征块（与 12 相同）

In [5]:
def read_raw_ltv(path):
    text = pd.read_csv(path, usecols=['loan_to_value_ratio'], dtype=str,
                       keep_default_na=False)['loan_to_value_ratio']
    return text


raw_ltv = pd.concat([read_raw_ltv(PREPARED_DIR / 'train_features.csv'),
                     read_raw_ltv(PREPARED_DIR / 'test_features.csv')], ignore_index=True)
both = pd.concat([X_base, X_test_base], ignore_index=True)
require(len(raw_ltv) == len(both), 'LTV 文本行数不符')
require(pd.to_numeric(raw_ltv.replace('', np.nan)).equals(both['loan_to_value_ratio'])
        or np.allclose(pd.to_numeric(raw_ltv.replace('', np.nan)), both['loan_to_value_ratio'],
                       equal_nan=True), 'LTV 文本与数值不一致')

decimals = raw_ltv.str.split('.').str[1].fillna('')
precision = np.select(
    [raw_ltv.eq(''), decimals.isin(['', '0']), decimals.str.len().eq(1),
     decimals.str.len().eq(2), decimals.str.len().eq(3)],
    ['missing', 'integer', 'dec1', 'dec2', 'dec3'], default='dec4plus',
)
ltv = both['loan_to_value_ratio']
band = np.select(
    [ltv.isna(), ltv < 80, ltv == 80, ltv < 95, ltv < 97, ltv < 100, ltv == 100],
    ['missing', 'lt80', 'eq80', '80_95', '95_97', '97_100', 'eq100'], default='gt100',
)
PROGRAM_MAX_LTV = {'1': 97.0, '2': 96.5, '3': 100.0, '4': 100.0}   # 常规/FHA/VA/RHS
unknown_types = set(both['loan_type']) - set(PROGRAM_MAX_LTV)
require(not unknown_types, f'未知 loan_type：{sorted(unknown_types)}')
implied_value = safe_ratio(both['loan_amount'] * 100, ltv)
term = both['loan_term']

round2 = {
    'precision': pd.DataFrame({'ltv_precision': precision}),
    'thresholds': pd.DataFrame({
        'ltv_band': band,
        'ltv_over_program': ltv - both['loan_type'].map(PROGRAM_MAX_LTV).astype('float64'),
    }),
    'implied': pd.DataFrame({
        'implied_property_value': implied_value,
        'implied_value_ratio': safe_ratio(implied_value, both['property_value']),
    }),
    'term': pd.DataFrame({
        'term_is_360': term.eq(360).astype('float64').where(term.notna()),
        'term_years': (term / 12).round(1),
    }),
}
n_train = len(X_base)
for name, frame in round2.items():
    require(len(frame) == len(both), f'{name} 行数不符')
    BLOCKS[name] = (frame.iloc[:n_train].reset_index(drop=True),
                    frame.iloc[n_train:].reset_index(drop=True))
NEW_CATEGORICAL |= {'ltv_precision', 'ltv_band'}

train_labels_check = y.to_numpy()
for column, frame in [('ltv_precision', round2['precision']), ('ltv_band', round2['thresholds'])]:
    table = pd.crosstab(frame[column].iloc[:n_train], train_labels_check, normalize='index')
    counts = frame[column].iloc[:n_train].value_counts()
    display(table.assign(rows=counts).round(3))
for name in ['implied', 'term']:
    display(BLOCKS[name][0].describe().T.round(3))
del both

col_0,0,1,rows
ltv_precision,,,
dec1,0.125,0.875,12131
dec2,0.156,0.844,9717
dec3,0.125,0.875,19531
dec4plus,0.154,0.846,306
integer,0.093,0.907,51441
missing,0.462,0.538,11874


col_0,0,1,rows
ltv_band,,,
80_95,0.122,0.878,14329
95_97,0.117,0.883,20177
97_100,0.129,0.871,6015
eq100,0.107,0.893,7295
eq80,0.081,0.919,13455
gt100,0.176,0.824,5711
lt80,0.096,0.904,26144
missing,0.462,0.538,11874


,count,mean,std,min,25%,50%,75%,max
implied_property_value,93126.0,526207.615,2740245.581,0.011,243523.316,367875.648,554999.602,6.270987e+08
implied_value_ratio,92643.0,1.051,2.960,0.003,0.989,1.000,1.009,1.108650e+02


,count,mean,std,min,25%,50%,75%,max
term_is_360,101693.0,0.841,0.366,0.0,1.0,1.0,1.0,1.0
term_years,101693.0,28.146,5.534,0.1,30.0,30.0,30.0,40.0


## 6. 折内预处理与 TabM-mini 模型

预处理器在每个外层折的训练部分上重新拟合，验证和测试只做变换。

In [6]:
nn_train, nn_test, nn_cats = variant_frames(G7)
nn_nums = [c for c in nn_train.columns if c not in nn_cats]
print(f'特征：{nn_train.shape[1]} 列 = {len(nn_nums)} 数值 + {len(nn_cats)} 类别')


class FoldPreprocessor:
    """只在外层训练部分上拟合：分位数正态化、分段线性编码 (PLE)、缺失标记、类别词表。"""

    def __init__(self, bins, rare_min_count):
        self.bins = bins
        self.rare_min_count = rare_min_count

    def fit(self, frame):
        num = frame[nn_nums].to_numpy(dtype='float64')
        self.quantile = QuantileTransformer(n_quantiles=1000, output_distribution='normal',
                                            subsample=10**9, random_state=SEED)
        self.quantile.fit(num)                      # sklearn 会忽略 NaN
        self.edges = []
        for j in range(num.shape[1]):
            values = num[:, j][~np.isnan(num[:, j])]
            edges = np.unique(np.quantile(values, np.linspace(0, 1, self.bins + 1)))
            self.edges.append(edges if len(edges) >= 2 else np.array([values.min(), values.min() + 1]))
        self.has_missing = np.isnan(num).any(axis=0)
        self.vocab = []
        for column in nn_cats:
            counts = frame[column].value_counts()
            kept = counts.index[counts >= self.rare_min_count]
            self.vocab.append({value: i + 1 for i, value in enumerate(kept)})   # 0 = 未知/稀有
        self.cardinalities = [len(v) + 1 for v in self.vocab]
        return self

    def transform(self, frame):
        num = frame[nn_nums].to_numpy(dtype='float64')
        missing = np.isnan(num)
        normal = np.nan_to_num(np.clip(self.quantile.transform(num), -5, 5), nan=0.0)
        ple_parts = []
        for j, edges in enumerate(self.edges):
            x = num[:, j:j + 1]
            left, right = edges[:-1][None, :], edges[1:][None, :]
            part = np.clip((x - left) / np.maximum(right - left, 1e-12), 0, 1)
            part = np.nan_to_num(part, nan=0.0)
            ple_parts.append(part)
        dense = np.concatenate([normal] + ple_parts + [missing[:, self.has_missing]], axis=1)
        cats = np.stack([frame[c].map(v).fillna(0).to_numpy(dtype='int64')
                         for c, v in zip(nn_cats, self.vocab)], axis=1)
        return dense.astype('float32'), cats


class TabMMini(nn.Module):
    """TabM-mini：k 个成员共享同一个 MLP，只在输入端各有一个随机 ±1 的缩放向量；预测取 k 个成员概率的平均。"""

    def __init__(self, d_dense, cardinalities, k, width, depth, dropout, cat_dim):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(c, cat_dim) for c in cardinalities])
        d_in = d_dense + cat_dim * len(cardinalities)
        self.adapter = nn.Parameter(torch.randint(0, 2, (k, d_in)).float() * 2 - 1)
        layers = []
        for i in range(depth):
            layers += [nn.Linear(d_in if i == 0 else width, width), nn.ReLU(), nn.Dropout(dropout)]
        self.backbone = nn.Sequential(*layers)
        self.head_weight = nn.Parameter(torch.randn(k, width) / width ** 0.5)
        self.head_bias = nn.Parameter(torch.zeros(k))

    def forward(self, dense, cats):
        parts = [dense] + [emb(cats[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(parts, dim=1)                       # (B, D)
        x = x[:, None, :] * self.adapter[None]            # (B, k, D)
        h = self.backbone(x)                              # (B, k, width)
        return (h * self.head_weight).sum(-1) + self.head_bias   # (B, k) logits


def predict(model, dense, cats, batch=4096):
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(dense), batch):
            d = torch.from_numpy(dense[start:start + batch]).to(DEVICE)
            c = torch.from_numpy(cats[start:start + batch]).to(DEVICE)
            out.append(torch.sigmoid(model(d, c)).mean(1).cpu().numpy())
    return np.concatenate(out).astype('float64')


def train_fold(fold):
    cfg = NN_CONFIG
    fit_idx = np.flatnonzero(fold_assignment != fold)
    valid_idx = np.flatnonzero(fold_assignment == fold)
    inner_fit, inner_stop = train_test_split(
        fit_idx, test_size=cfg['inner_stop_fraction'], random_state=SEED, stratify=y.iloc[fit_idx])
    prep = FoldPreprocessor(cfg['ple_bins'], cfg['rare_min_count']).fit(nn_train.iloc[fit_idx])
    data = {name: prep.transform(frame) for name, frame in [
        ('fit', nn_train.iloc[inner_fit]), ('stop', nn_train.iloc[inner_stop]),
        ('valid', nn_train.iloc[valid_idx]), ('test', nn_test)]}
    y_fit = torch.from_numpy(y.iloc[inner_fit].to_numpy(dtype='float32')).to(DEVICE)
    y_stop = y.iloc[inner_stop].to_numpy()

    torch.manual_seed(SEED + fold)
    model = TabMMini(data['fit'][0].shape[1], prep.cardinalities, cfg['k'], cfg['width'],
                     cfg['depth'], cfg['dropout'], cfg['cat_dim']).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    loss_fn = nn.BCEWithLogitsLoss()
    dense_fit = torch.from_numpy(data['fit'][0]).to(DEVICE)
    cats_fit = torch.from_numpy(data['fit'][1]).to(DEVICE)
    best_auc, best_state, best_epoch, waited = -1.0, None, 0, 0
    started = time.perf_counter()
    for epoch in range(1, cfg['max_epochs'] + 1):
        model.train()
        order = torch.randperm(len(dense_fit), device=DEVICE)
        for start in range(0, len(order), cfg['batch_size']):
            batch = order[start:start + cfg['batch_size']]
            logits = model(dense_fit[batch], cats_fit[batch])
            loss = loss_fn(logits, y_fit[batch][:, None].expand_as(logits))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        stop_auc = roc_auc_score(y_stop, predict(model, *data['stop']))
        if stop_auc > best_auc:
            best_auc, best_epoch, waited = stop_auc, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
        print(f'  fold {fold} epoch {epoch:2d}: 内层早停集 AUC={stop_auc:.5f}'
              f'{"  *" if waited == 0 else ""}', flush=True)
        if waited >= cfg['patience']:
            break
    model.load_state_dict(best_state)
    valid_prob = predict(model, *data['valid'])
    test_prob = predict(model, *data['test'])
    require(np.isfinite(valid_prob).all() and np.isfinite(test_prob).all(), '神经网络预测含非有限值')
    actual = y.iloc[valid_idx]
    stem = OUTPUT_DIR / f'TabMMini__fold{fold}'
    pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                  'prob': valid_prob}).to_csv(f'{stem}__valid.csv', index=False, float_format='%.12f')
    pd.DataFrame({'id': test_ids.to_numpy(), 'accepted': test_prob}).to_csv(
        f'{stem}__test.csv', index=False, float_format='%.12f')
    result = {
        'group': 'TabMMini', 'fold': fold, 'config': cfg, 'blocks': G7,
        'dense_dim': int(data['fit'][0].shape[1]), 'categorical_columns': len(nn_cats),
        'best_epoch': best_epoch, 'inner_stop_auc': best_auc,
        'valid_auc': roc_auc_score(actual, valid_prob),
        'valid_logloss': log_loss(actual, valid_prob, labels=[0, 1]),
        'seconds': time.perf_counter() - started, 'device': DEVICE,
        'torch': torch.__version__, 'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    write_json(Path(f'{stem}.json'), result)
    del model, dense_fit, cats_fit
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return result

特征：72 列 = 26 数值 + 46 类别


## 7. 训练（每折结束即保存，可续跑）

In [7]:
fixed_config(OUTPUT_DIR, {
    'purpose': 'TabM-mini (PyTorch) on G7 features; inner 15% early stop, no outer refit',
    'seed': SEED, 'nn_config': NN_CONFIG, 'blocks': G7, 'data_hashes': DATA_HASHES,
})
for fold in FOLDS:
    path = OUTPUT_DIR / f'TabMMini__fold{fold}.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        print(f'TabMMini fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
        continue
    result = train_fold(fold)
    print(f'TabMMini fold {fold}: AUC={result["valid_auc"]:.6f}  最佳 epoch={result["best_epoch"]}  '
          f'{result["seconds"]/60:.1f} 分钟', flush=True)

  fold 1 epoch  1: 内层早停集 AUC=0.90448  *
  fold 1 epoch  2: 内层早停集 AUC=0.91148  *
  fold 1 epoch  3: 内层早停集 AUC=0.91367  *
  fold 1 epoch  4: 内层早停集 AUC=0.91592  *
  fold 1 epoch  5: 内层早停集 AUC=0.91676  *
  fold 1 epoch  6: 内层早停集 AUC=0.91760  *
  fold 1 epoch  7: 内层早停集 AUC=0.91808  *
  fold 1 epoch  8: 内层早停集 AUC=0.91825  *
  fold 1 epoch  9: 内层早停集 AUC=0.91884  *
  fold 1 epoch 10: 内层早停集 AUC=0.91909  *
  fold 1 epoch 11: 内层早停集 AUC=0.91751
  fold 1 epoch 12: 内层早停集 AUC=0.91833
  fold 1 epoch 13: 内层早停集 AUC=0.91746
  fold 1 epoch 14: 内层早停集 AUC=0.91668
  fold 1 epoch 15: 内层早停集 AUC=0.91845
  fold 1 epoch 16: 内层早停集 AUC=0.91670
  fold 1 epoch 17: 内层早停集 AUC=0.91624
  fold 1 epoch 18: 内层早停集 AUC=0.91691
TabMMini fold 1: AUC=0.908783  最佳 epoch=10  1.4 分钟


## 8. 与 CatBoost（15 的 B_seeds）在同一折上比较与融合

只用已完成的折。`delta_wXX` 为「(1−w)·CatBoost + w·TabM」相对 CatBoost 的 AUC 变化。

In [8]:
nn_done = sorted(json.loads(p.read_text(encoding='utf-8'))['fold']
                 for p in OUTPUT_DIR.glob('TabMMini__fold?.json'))
require(nn_done, '神经网络还没有完成的折')

CAT_BASES = ['M1_C11_G7', 'M2_C7_G7', 'M3_C11_lean']
SEED_SUFFIXES = ['', '_s2027', '_s31415']
cat_groups = [b + s for b in CAT_BASES for s in SEED_SUFFIXES]
missing = [g for g in cat_groups for k in range(1, 6)
           if not (CATBOOST_DIR / f'{g}__fold{k}__valid.csv').is_file()]
require(not missing, f'缺少 15 的 CatBoost 结果，例如 {missing[:3]}；请先完成 15')


def catboost_fold(fold):
    """15 的最终方案 B_seeds：每个基础组先对 3 个种子平均，再三组等权。"""
    per_base = []
    for base in CAT_BASES:
        frames = [pd.read_csv(CATBOOST_DIR / f'{base}{s}__fold{fold}__valid.csv') for s in SEED_SUFFIXES]
        for f in frames[1:]:
            require(f['id'].equals(frames[0]['id']), 'CatBoost OOF 行顺序不一致')
        per_base.append(np.mean([f['prob'].to_numpy() for f in frames], axis=0))
    return frames[0][['id', 'accepted']].assign(cat=np.mean(per_base, axis=0))


rows = []
for fold in nn_done:
    nn_valid = pd.read_csv(OUTPUT_DIR / f'TabMMini__fold{fold}__valid.csv')
    merged = catboost_fold(fold).merge(nn_valid[['id', 'prob']], on='id', validate='one_to_one')
    require(len(merged) == len(nn_valid), '神经网络与 CatBoost 的验证行不一致')
    row = {'fold': fold,
           'catboost_auc': roc_auc_score(merged['accepted'], merged['cat']),
           'tabm_auc': roc_auc_score(merged['accepted'], merged['prob']),
           'corr': np.corrcoef(merged['cat'], merged['prob'])[0, 1]}
    for w in BLEND_WEIGHTS:
        row[f'blend_w{int(w * 100)}'] = roc_auc_score(merged['accepted'],
                                                     (1 - w) * merged['cat'] + w * merged['prob'])
    rows.append(row)
table = pd.DataFrame(rows).set_index('fold')
for w in BLEND_WEIGHTS:
    table[f'delta_w{int(w * 100)}'] = table[f'blend_w{int(w * 100)}'] - table['catboost_auc']
display(table.round(6))
summary = table.mean().to_frame('mean').T
display(summary.round(6))
table.to_csv(OUTPUT_DIR / 'blend_by_fold.csv')

gap = summary['catboost_auc'].iloc[0] - summary['tabm_auc'].iloc[0]
best_w = max(BLEND_WEIGHTS, key=lambda w: summary[f'delta_w{int(w * 100)}'].iloc[0])
best_delta = summary[f'delta_w{int(best_w * 100)}'].iloc[0]
print(f'TabM 单独比 CatBoost 低 {gap:.5f}；与 CatBoost 的相关系数 {summary["corr"].iloc[0]:.4f}；'
      f'最好的融合权重 {best_w}：{best_delta:+.6f}')
if len(nn_done) < 5:
    verdict = ('值得跑完五折：把 FOLDS 改为 [1, 2, 3, 4, 5] 后从头运行（第 1 折会自动跳过）。'
               if best_delta >= 0.0003 else
               '第 1 折上融合提升不足 0.0003，建议到此为止，保持 15 的 B_seeds 作为最终提交。')
    print('判断：' + verdict)
else:
    better = int((table[f'delta_w{int(best_w * 100)}'] > 0).sum())
    print(f'五折：权重 {best_w} 时 {better}/5 折优于 CatBoost。')
    cat_test = []
    for base in CAT_BASES:
        tests = [pd.read_csv(CATBOOST_DIR / f'{base}{s}__fold{k}__test.csv', dtype={'id': str})
                 for s in SEED_SUFFIXES for k in range(1, 6)]
        for t in tests:
            require(t['id'].equals(sample_submission['id']), 'CatBoost 测试 ID 顺序不一致')
        cat_test.append(np.mean([t['accepted'].to_numpy() for t in tests], axis=0))
    cat_test = np.mean(cat_test, axis=0)
    nn_tests = [pd.read_csv(OUTPUT_DIR / f'TabMMini__fold{k}__test.csv', dtype={'id': str})
                for k in range(1, 6)]
    for t in nn_tests:
        require(t['id'].equals(sample_submission['id']), 'TabM 测试 ID 顺序不一致')
    nn_test = np.mean([t['accepted'].to_numpy() for t in nn_tests], axis=0)
    for w in BLEND_WEIGHTS:
        path = OUTPUT_DIR / f'submission_Bseeds_TabM_w{int(w * 100)}.csv'
        pd.DataFrame({'id': sample_submission['id'],
                      'accepted': (1 - w) * cat_test + w * nn_test}).to_csv(
            path, index=False, float_format='%.12f')
        print('写出', path.name)
    print('只有当某个权重五折平均 ≥ +0.0003 且 ≥ 4/5 折更好时，才替换 15 的 B_seeds。')

,catboost_auc,tabm_auc,corr,blend_w10,blend_w20,blend_w30,delta_w10,delta_w20,delta_w30
fold,,,,,,,,,
1,0.914892,0.908783,0.983936,0.914879,0.914701,0.9144,-0.000013,-0.00019,-0.000492


,catboost_auc,tabm_auc,corr,blend_w10,blend_w20,blend_w30,delta_w10,delta_w20,delta_w30
mean,0.914892,0.908783,0.983936,0.914879,0.914701,0.9144,-0.000013,-0.00019,-0.000492


TabM 单独比 CatBoost 低 0.00611；与 CatBoost 的相关系数 0.9839；最好的融合权重 0.1：-0.000013
判断：第 1 折上融合提升不足 0.0003，建议到此为止，保持 15 的 B_seeds 作为最终提交。
